In [3]:
import numpy as np
import math

# Dummy data for testing (m=5 examples)
y = np.array([1, 0, 1, 1, 0])                     # True labels
y_hat = np.array([0.9, 0.1, 0.8, 0.95, 0.2])      # Predicted probabilities
z = np.array([2.5, -1.0, 0.5, 3.0, -2.2])         # Raw logits

**Mean Squared Error (MSE)**
    MSE measures the average squared difference between the estimated values and the actual value.Equation:$$MSE = \frac{1}{m} \sum_{i=1}^{m} (y^{(i)} - \hat{y}^{(i)})^2$$

In [14]:
                                #Vertorized Implementation

#my implementation
m = len(y)
mse = (1/m)*np.sum((y-y_hat)**2)
print("mean square error", mse)


# optimized version
def mse_vec(y, y_hat):
    # Vectorized: NumPy handles the element-wise subtraction and squaring instantly
    return np.mean((y - y_hat)**2)
    
print("actual formula", np.mean((y - y_hat)**2))


                            #For loop implemenation

def mse_loop(y, y_hat):
    m = len(y)
    total_error = 0.0
    for i in range(m):
        total_error += (y[i] - y_hat[i])**2
    return total_error / m

# Assertion test
assert np.allclose(mse_loop(y, y_hat), mse_vec(y, y_hat))
print("MSE verified!", mse_loop(y, y_hat), mse_vec(y, y_hat))


mean square error 0.020499999999999997
actual formula 0.020499999999999997
MSE verified! 0.020499999999999997 0.020499999999999997


 **Sigmoid Function** \ <br><br>The sigmoid function maps any real number to a value between 0 and 1, making it perfect for predicting probabilities.<br>Equation:$$\sigma(z) = \frac{1}{1 + e^{-z}}$$


In [17]:
def sigmoid_loop(z):
    # We must pre-allocate an array of the same size when using loops
    result = np.zeros(len(z))
    for i in range(len(z)):
        result[i] = 1 / (1 + math.exp(-z[i]))
    return result

def sigmoid_vec(z):
    # Vectorized: np.exp applies the exponential function to every element in parallel
    return 1 / (1 + np.exp(-z))

# Assertion test
assert np.allclose(sigmoid_loop(z), sigmoid_vec(z))
print("Sigmoid verified!")

Sigmoid verified!


**Binary Cross-Entropy Cost Function** <br><br>Also known as log loss, this is the standard cost function for logistic regression. It heavily penalizes confident but incorrect predictions.<br><br>Equation:$$J = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

In [35]:
def bce_loop(y, y_hat):
    m = len(y)
    total_cost = 0.0
    for i in range(m):
        # math.log calculates the natural logarithm (base e)
        total_cost += y[i] * math.log(y_hat[i]) + (1 - y[i]) * math.log(1 - y_hat[i])
    return -total_cost / m

def bce_vec(y, y_hat):
    # Vectorized: Element-wise multiplication (*) and np.log for arrays
    return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

# Assertion test
assert np.allclose(bce_loop(y, y_hat), bce_vec(y, y_hat))
print("Binary Cross-Entropy verified!")

Binary Cross-Entropy verified!


# NumPy Broadcasting Cheatsheet

### 📖 Definition of Broadcasting
**Broadcasting** is NumPy's mechanism for performing arithmetic operations on arrays with different shapes. Instead of rewriting or manually duplicating data in memory (which is slow and inefficient), NumPy virtually stretches or replicates the smaller array along its missing or sizing dimensions so that the arrays match structural requirements for element-wise operations.

---

### 📜 The 2 Core Rules of Broadcasting
When operating on two arrays (`A` and `B`), NumPy compares their shapes **from right to left** (starting with the trailing dimensions). 

Two dimensions are compatible **only if**:
1. **They are exactly equal in size.**
2. **One of them is exactly 1.**

> **The Padding Rule:** If one array has fewer dimensions than the other, NumPy automatically pads the shorter shape on the **left** side with `1`s until they have the exact same number of dimensions.

---

### 🧠 The Mental Shortcut
To instantly know if any two shapes will broadcast, execute this 3-step mental algorithm:
1. **Align Right-to-Left:** Write the shapes down, aligning them perfectly on the right-hand side.
2. **Look for Disagreements:** Move left column by column. If the numbers in a column don't match, look for the **"Magic 1"**.
3. **The Test:** If they disagree and **neither** number is a `1`, it **fails**. If at least one number is a `1`, it **passes** (the `1` stretches to match the larger number).

---


In [52]:
import numpy as np

print("--- 1. THE PERFECT COMPATIBLE MATCH ---")
# Concept: (4, 1, 2, 5) and (3, 1, 5) -> Padded to (4, 1, 2, 5) and (1, 3, 1, 5)
a = np.random.randint(1, 7, size=(4, 1, 2, 5))
b = np.random.randint(1, 7, size=(3, 1, 5))

# Element-wise addition works perfectly via broadcasting
result_add = a + b
print(f"Array 'a' shape: {a.shape}")
print(f"Array 'b' shape: {b.shape}")
print(f"Result (a + b) shape: {result_add.shape}\n")


print("--- 2. THE RIGID SHAPE CLASH (FAIL) ---")
# Concept: (4, 3) and (4, 2) -> 3 vs 2 rightmost mismatch, neither is 1.
try:
    x = np.ones((4, 3))
    y = np.ones((4, 2))
    clash = x + y
except ValueError as e:
    print(f"Caught expected error: {e}\n")


print("--- 3. THE np.sum() TRAP VS CORRECT SOLUTION ---")
# Scenario: Stacking arrays in np.sum([a, b]) fails because list items must be rigid grids.
try:
    np.sum([a, b])
except ValueError as e:
    print(f"np.sum([a, b]) FAILED: Stacking raw shapes together ignores broadcasting parameters.")

# Correct Way: Broadcast FIRST using the operator, then sum the collapsed elements.
total_sum = np.sum(a + b)
print(f"SUCCESS: np.sum(a + b) computed a single scalar value: {total_sum}")


--- 1. THE PERFECT COMPATIBLE MATCH ---
Array 'a' shape: (4, 1, 2, 5)
Array 'b' shape: (3, 1, 5)
Result (a + b) shape: (4, 3, 2, 5)

--- 2. THE RIGID SHAPE CLASH (FAIL) ---
Caught expected error: operands could not be broadcast together with shapes (4,3) (4,2) 

--- 3. THE np.sum() TRAP VS CORRECT SOLUTION ---
np.sum([a, b]) FAILED: Stacking raw shapes together ignores broadcasting parameters.
SUCCESS: np.sum(a + b) computed a single scalar value: 789


In [46]:
import random

# Generates an integer from 1 to 6 (like rolling a die)
a = np.random.randint(1, 7, size=(4, 1, 2, 5))
b = np.random.randint(1, 7, size=(3, 1, 5))
print(a)
print(b)
total_sum = np.sum(a + b)
print(total_sum)

[[[[1 1 6 3 6]
   [4 3 1 5 1]]]


 [[[4 6 4 4 5]
   [3 3 4 4 1]]]


 [[[1 4 1 5 1]
   [5 5 6 5 6]]]


 [[[5 3 3 6 1]
   [6 5 6 5 4]]]]
[[[4 6 2 2 2]]

 [[2 4 3 5 4]]

 [[3 4 1 4 3]]]
